In [1]:
from typing import (
    TypedDict,
    Literal,
    Annotated,
)

import operator
import os
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    AIMessage,
    HumanMessage
)
from langchain_core.tools import tool
# from langchain.agents import create_agent

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.types import (
    Command,
    Send,
)
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found - add it to your .env file.")
    
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    max_retries=3,   # free models hit 429 rate limits often
    timeout=120,
)

In [4]:
# agent state

class AgentState(TypedDict):
    Task: str
    messages: Annotated[list, add_messages]
    output: str
    route: str

In [5]:
# creating Specialist agents for Frontend teams

# react agent
def react_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a React specialist. Handle all React component and frontend tasks.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# ui agent
def ui_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a UI specialist. Build clean, accessible user interfaces and handle all UI releated task.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# Accessibility Agent
def accessibility_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Accessibility specialist. Ensure all UI components meet WCAG 2.1 AA standards: semantic HTML, keyboard navigation, screen reader support, color contrast, and proper ARIA usage.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

In [6]:
# creating Specialist agents for backend teams

# api agent
def api_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an API specialist. Design RESTful APIs, handle routing, validation, serialization, and ensure consistent error responses.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# auth agent
def auth_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Authentication specialist. Implement secure auth: JWT/OAuth2, token refresh, password hashing, MFA, session management, and RBAC.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# microservice agent
def microservice_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a Microservice specialist. Design and review microservice architectures, service boundaries, inter-service communication, resilience, and deployment patterns based on the requirements provided.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

In [7]:
# creating Specialist agents for database teams

# database agents
def sql_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL specialist. Help with database and SQL tasks by producing accurate, efficient, and secure queries. Use only the schema and requirements provided; if essential details are missing, ask a concise clarifying question. Briefly explain assumptions when needed, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

def schema_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert database schema designer. Help create, review, and improve database schemas based on the requirements provided. Recommend appropriate tables, columns, data types, primary and foreign keys, constraints, and relationships. Prioritize data integrity, clarity, and maintainability. Ask a concise clarifying question if essential requirements are missing, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

def query_optimization_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL query optimization specialist. Analyze SQL queries and the provided schema to identify ways to improve performance without changing the results. Consider indexes, joins, filtering, query plans, and database-specific behavior. State any assumptions, ask a concise clarifying question if essential details are missing, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [8]:
# creating Specialist agents for QA teams

# unit test Agent
def unit_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert unit testing specialist. Help write focused tests for individual functions, methods, and components. Cover expected behavior, edge cases, and error conditions using the project’s existing testing framework and conventions. Keep tests deterministic and independent. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# Integration test Agent
def integration_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert integration testing specialist. Help test how multiple components, services, or modules work together. Follow the project’s existing testing framework and conventions, and cover key data flows, boundaries, and failure cases. Keep tests reliable and avoid unnecessary external dependencies. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# regresion Agent
def regression_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert regression testing specialist. Help create tests that catch previously fixed bugs and prevent existing behavior from breaking. Use the provided bug details, expected behavior, and project testing conventions to write focused, repeatable tests. If essential context is missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [9]:
# creating Specialist agents for DevOps teams

# Docker Agent
def docker_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Docker specialist. Help create, review, and troubleshoot Dockerfiles, images, and Compose configurations. Recommend secure, efficient, and reproducible container practices, following the project’s requirements and conventions. If essential details are missing, ask a concise clarifying question; otherwise return only the requested configuration or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# Kubernetes Agent
def kubernetes_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Kubernetes specialist. Help design, write, review, and troubleshoot Kubernetes resources such as Deployments, Services, ConfigMaps, and Ingresses. Prioritize reliability, security, and maintainability, and use the requirements and environment details provided. If essential information is missing, ask a concise clarifying question; otherwise return only the requested manifests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# CI/CD Agent
def cicd_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert CI/CD specialist. Help design, write, and troubleshoot continuous integration and deployment pipelines using the platform and project requirements provided. Focus on reliable automation, clear validation steps, secure handling of secrets, and safe deployments. If essential details are missing, ask a concise clarifying question; otherwise return only the requested pipeline configuration or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [10]:
# creating Specialist agents for security Team

# Vulnerability Agent
def vulnerability_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert application security specialist. Review the provided code or configuration for potential vulnerabilities. For each finding, describe the affected area, likely impact, severity, and a safe remediation. Focus on defensive analysis and do not expose secrets or provide instructions for exploiting systems. If essential context is missing, ask a concise clarifying question; otherwise return only the requested findings or guidance.")
    response = model.invoke([system] + state['messages'])
    return {'messages': [response], 'output': response.content}
# IAM Agent
def iam_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Identity and Access Management (IAM) specialist. Help design, review, and troubleshoot identities, roles, permissions, and access policies. Apply least privilege, separation of duties, and secure authentication practices. Explain risks and suggest practical improvements based on the requirements provided. If essential details are missing, ask a concise clarifying question; otherwise return only the requested policy or guidance.")
    response = model.invoke([system] + state["messages"])
    return {'messages': [response], 'output': response.content}
# Secrets Agent
def secrets_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert secrets-management specialist. Help identify and prevent unsafe handling of credentials, tokens, keys, and other sensitive values. Recommend secure storage, access controls, rotation, and safe injection into applications and pipelines. Never repeat or expose secret values; redact them if they appear in the provided material. If essential details are missing, ask a concise clarifying question; otherwise return only the requested findings or guidance.")
    response = model.invoke([system] + state["messages"])
    return {'messages': [response], 'output': response.content}

In [11]:
# frontend route
class FrontendRoute(BaseModel):
    next: Literal["react_agent", "ui_agent", "accessibility_agent", "FINISH"]

def frontend_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""You are the Frontend Team Supervisor.
Route the task to the correct specialist:
- react_agent: React components, hooks, state management
- ui_agent: UI layout, styling, visual design
- accessibility_agent: WCAG compliance, ARIA, keyboard navigation
- FINISH: task is complete, no more routing needed

Return only the agent name.""")

    structured_model = model.with_structured_output(FrontendRoute)
    result = structured_model.invoke([system] + state['messages'])
    return {"route": result.next}
# route to frontend agent
def route_frontend(state: AgentState) -> str:
    return state["route"]

# frontend graph
frontend_team_subgraph = StateGraph(AgentState)

frontend_team_subgraph.add_node("frontend_supervisor", frontend_team_supervisor)
frontend_team_subgraph.add_node("react_agent", react_agent)
frontend_team_subgraph.add_node("ui_agent", ui_agent)
frontend_team_subgraph.add_node("accessibility_agent", accessibility_agent)

frontend_team_subgraph.add_edge(START, "frontend_supervisor")

frontend_team_subgraph.add_conditional_edges(
    "frontend_supervisor",
    route_frontend,
    {
        "react_agent": "react_agent",
        "ui_agent": "ui_agent",
        "accessibility_agent": "accessibility_agent",
        "FINISH": END
    }
)

# after each specialist finishes → back to supervisor
frontend_team_subgraph.add_edge("react_agent", "frontend_supervisor")
frontend_team_subgraph.add_edge("ui_agent", "frontend_supervisor")
frontend_team_subgraph.add_edge("accessibility_agent", "frontend_supervisor")

frontend_team = frontend_team_subgraph.compile()

In [12]:
# backend route

class BackendRoute(BaseModel):
    next: Literal['api_agent','auth_agent','microservice_agent','FINISH']

def backend_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Backend Team Supervisor.
Route the task to the correct specialist:
- API Agent: API design, endpoints, request/response handling
- Auth Agent: authentication, authorization, identity flows
- Microservice Agent: microservice architecture, service communication, distributed systems
- FINISH: task is complete, no more routing needed

Return only the agent name.
""")
    structured_model = model.with_structured_output(BackendRoute)
    result = structured_model.invoke([system]+state['messages'])
    return {'route':result.next}

def route_backend(state: AgentState) -> str:
    state['route']

backend_team_subgraph = StateGraph(AgentState)

backend_team_subgraph.add_node("backend_supervisor", backend_team_supervisor)
backend_team_subgraph.add_node("api_agent", api_agent)
backend_team_subgraph.add_node("auth_agent", auth_agent)
backend_team_subgraph.add_node("microservice_agent", microservice_agent)

backend_team_subgraph.add_edge(START, "backend_supervisor")

backend_team_subgraph.add_conditional_edges(
    "backend_supervisor",
    route_backend,
    {
        "api_agent": "api_agent",
        "auth_agent": "auth_agent",
        "microservice_agent": "microservice_agent",
        "FINISH": END
    }
)

# after each specialist finishes → back to supervisor
backend_team_subgraph.add_edge("api_agent", "backend_supervisor")
backend_team_subgraph.add_edge("auth_agent", "backend_supervisor")
backend_team_subgraph.add_edge("microservice_agent", "backend_supervisor")

backend_team = backend_team_subgraph.compile()

In [13]:
# database route

class DatabaseRoute(BaseModel):
    next: Literal['sql_agent','schema_agent','query_optimization_agent','FINISH']

def database_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Database Team Supervisor.
Route the task to the correct specialist:
- SQL Agent: SQL statements, database operations, stored procedures
- Schema Agent: database schemas, tables, relationships, migrations
- Query Optimization Agent: query performance, indexing, execution plans
- FINISH: task is complete, no more routing needed

Return only the agent name.
""")
    structured_model = model.with_structured_output(DatabaseRoute)
    result = structured_model.invoke([system]+state['messages'])
    return {'route':result.next}

def route_database(state: AgentState) -> str:
    state['route']

database_team_subgraph = StateGraph(AgentState)

database_team_subgraph.add_node("database_supervisor", database_team_supervisor)
database_team_subgraph.add_node("sql_agent", sql_agent)
database_team_subgraph.add_node("schema_agent", schema_agent)
database_team_subgraph.add_node("Query_optimization_agent", query_optimization_agent)

database_team_subgraph.add_edge(START, "database_supervisor")

database_team_subgraph.add_conditional_edges(
    "database_supervisor",
    route_database,
    {
        "sql_agent": "sql_agent",
        "schema_agent": "schema_agent",
        "Query_optimization_agent": "Query_optimization_agent",
        "FINISH": END
    }
)

# after each specialist finishes → back to supervisor
database_team_subgraph.add_edge("sql_agent", "database_supervisor")
database_team_subgraph.add_edge("schema_agent", "database_supervisor")
database_team_subgraph.add_edge("Query_optimization_agent", "database_supervisor")

database_team = database_team_subgraph.compile()

In [14]:
# QA route

class QaRoute(BaseModel):
    next: Literal['unit_test_agent','integration_test_agent','regression_agent','FINISH']

def qa_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the QA Team Supervisor.
Route the task to the correct specialist:
- Unit Test Agent: unit tests and isolated component or function testing
- Integration Test Agent: integration tests and interactions between components
- Regression Agent: regression testing and verification of existing behavior
- FINISH: task is complete, no more routing needed

Return only the agent name.
""")
    structure_model = model.with_structured_output(QaRoute)
    result = structure_model.invoke([system]+state['messages'])
    return {'route': result.next}

def route_qa(state: AgentState) -> str:
    state['route']

# Qa sub-graph
qa_team_subgraph = StateGraph(AgentState)
qa_team_subgraph.add_node('qa_supervisor',qa_team_supervisor)
qa_team_subgraph.add_node('unit_test_agent', unit_test_agent)
qa_team_subgraph.add_node('integration_test_agent',integration_test_agent)
qa_team_subgraph.add_node('regression_agent', regression_agent)

qa_team_subgraph.add_edge(START, 'qa_supervisor')
qa_team_subgraph.add_conditional_edges(
    'qa_supervisor',
    route_qa,
    {
        'unit_test_agent': 'unit_test_agent',
        'integration_test_agent': 'integration_test_agent',
        'regression_agent': 'regression_agent',
        'FINISH': END
    }
)

qa_team_subgraph.add_edge('unit_test_agent', 'qa_supervisor')
qa_team_subgraph.add_edge('integration_test_agent', 'qa_supervisor')
qa_team_subgraph.add_edge('regression_agent', 'qa_supervisor')

qa_team = qa_team_subgraph.compile()

In [15]:
# DevOps route

class DevOpsRoute(BaseModel):
    next: Literal['docker_agent','kubernetes_agent','cicd_agent','FINISH']

def devops_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the DevOps Team Supervisor.
Route the task to the correct specialist:
- Docker Agent: Dockerfiles, container images, and container configuration
- Kubernetes Agent: Kubernetes resources, deployments, and cluster configuration
- CI/CD Agent: build pipelines, automated testing, and deployment workflows
- FINISH: task is complete, no more routing needed

Return only the agent name.
""")
    structure_model = model.with_structured_output(DevOpsRoute)
    result = structure_model.invoke([system]+state['messages'])
    return {'route': result.next}

def route_devops(state: AgentState) -> str:
    state['route']

# Qa sub-graph
devops_team_subgraph = StateGraph(AgentState)
devops_team_subgraph.add_node('devops_supervisor',devops_team_supervisor)
devops_team_subgraph.add_node('docker_agent', docker_agent)
devops_team_subgraph.add_node('kubernetes_agent',kubernetes_agent)
devops_team_subgraph.add_node('cicd_agent', cicd_agent)

devops_team_subgraph.add_edge(START, 'devops_supervisor')
devops_team_subgraph.add_conditional_edges(
    'devops_supervisor',
    route_devops,
    {
        'docker_agent': 'docker_agent',
        'kubernetes_agent': 'kubernetes_agent',
        'cicd_agent': 'cicd_agent',
        'FINISH': END
    }
)

devops_team_subgraph.add_edge('docker_agent', 'devops_supervisor')
devops_team_subgraph.add_edge('kubernetes_agent', 'devops_supervisor')
devops_team_subgraph.add_edge('cicd_agent', 'devops_supervisor')

devops_team = devops_team_subgraph.compile()

In [16]:
# Security route

class SecurityRoute(BaseModel):
    next: Literal['vulnerability_agent','iam_agent','secrets_agent','FINISH']

def security_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Security Team Supervisor.
Route the task to the correct specialist:
- Vulnerability Agent: vulnerability assessment, secure coding, and remediation
- IAM Agent: identity and access management, permissions, and access controls
- Secrets Agent: secret storage, handling, rotation, and exposure prevention
- FINISH: task is complete, no more routing needed

Return only the agent name.
""")
    structure_model = model.with_structured_output(SecurityRoute)
    result = structure_model.invoke([system]+state['messages'])
    return {'route': result.next}

def route_security(state: AgentState) -> str:
    state['route']

# Qa sub-graph
security_team_subgraph = StateGraph(AgentState)
security_team_subgraph.add_node('security_supervisor',security_team_supervisor)
security_team_subgraph.add_node('vulnerability_agent', vulnerability_agent)
security_team_subgraph.add_node('iam_agent',iam_agent)
security_team_subgraph.add_node('secrets_agent', secrets_agent)

security_team_subgraph.add_edge(START, 'security_supervisor')
security_team_subgraph.add_conditional_edges(
    'security_supervisor',
    route_security,
    {
        'vulnerability_agent': 'vulnerability_agent',
        'iam_agent': 'iam_agent',
        'secrets_agent': 'secrets_agent',
        'FINISH': END
    }
)

security_team_subgraph.add_edge('vulnerability_agent', 'security_supervisor')
security_team_subgraph.add_edge('iam_agent', 'security_supervisor')
security_team_subgraph.add_edge('secrets_agent', 'security_supervisor')

security_team = security_team_subgraph.compile()

In [ ]:
# top supervisor
class route_to(BaseModel):
    next: Literal['frontend_team','backend_team','database_team','qa_team', 'devops_team', 'security_team', 'FINISH']

def supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Top-Level Engineering Supervisor.
Analyze the task and route it to the team best suited to handle it:

- Frontend Team: React components, hooks, state management, UI layout, styling, visual design, accessibility, ARIA, and keyboard navigation
- Backend Team: APIs, authentication and authorization, and microservices
- Database Team: SQL, database schemas, migrations, and query optimization
- QA Team: unit, integration, and regression testing
- DevOps Team: Docker, Kubernetes, CI/CD, builds, and deployment workflows
- Security Team: vulnerability assessment, identity and access management, and secrets handling
- FINISH: the task is complete and no further routing is needed

Choose the most relevant team based on the task. If it spans multiple areas, route it to the team responsible for the primary work.

Return only one of these exact names:
Frontend Team
Backend Team
Database Team
QA Team
DevOps Team
Security Team
FINISH
""")
    structure_model = model.with_structured_output(route_to)
    result = structure_model.invoke([system]+state['messages'])
    return {'route': result.next}

def route_team(state: AgentState) -> str:
    state['route']

builder = StateGraph(AgentState)
builder.add_node('supervisor', supervisor)
builder.add_node('frontend_team', frontend_team)
builder.add_node('backend_team', backend_team)
builder.add_node('database_team', database_team)
builder.add_node('qa_team', qa_team)
builder.add_node('devops_team', devops_team)
builder.add_node('security_team', security_team)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges(
    'supervisor',
    route_team,
    {
        'frontend_team':'frontend_team',
        'backend_team':'backend_team',
        'database_team':'database_team',
        'qa_team':'qa_team',
        'devops_team':'devops_team',
        'security_team':'security_team',
        'FINISH': END
    })

builder.add_edge('frontend_team', 'supervisor')
builder.add_edge('backend_team', 'supervisor')
builder.add_edge('database_team', 'supervisor')
builder.add_edge('qa_team', 'supervisor')
builder.add_edge('devops_team', 'supervisor')
builder.add_edge('security_team', 'supervisor')
